In [1]:
# 1. Cấu hình — chỉ sửa tại đây
import argparse
import copy
import gc
import hashlib
import json
import math
import os
import re
import zipfile
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from IPython.display import display
from scipy.stats import binomtest
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer
from transformers.models.roberta.modeling_roberta import RobertaLayer

DATA_ROOT = Path('/kaggle/input/datasets/lamky690/refcoco/refcoco_data')
PERSON_JSON = DATA_ROOT / 'person_samples.json'
IMAGE_ROOT = DATA_ROOT / 'images'
DETR_CHECKPOINT = Path('/kaggle/working/pretrained/detr-r50-unc.pth')
OUT_DIR = Path('/kaggle/working/rec_eval')
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (488, 588)  # H, W; giữ đúng thiết lập eval trước đây
MAX_TEXT_LENGTH = 40
BATCH_SIZE = 16
FORCE_EVAL = False
EVAL_VERSION = 'rec-compact-v1'
EXPECTED_COUNTS = {'val': 5498, 'testA': 5273}
EXPECTED_TESTA_COLOR = 1328
SAME_SEED_ONLY = False  # False: 4 cặp chéo × 2 tập = 8 phép thử
SKIP_ALPHA = 0.0        # loader gán hệ số thật từ checkpoint

COLOR_WORDS = {
    'red', 'blue', 'green', 'yellow', 'black', 'white',
    'gray', 'grey', 'brown', 'pink', 'orange', 'purple',
    }
MODEL_PATHS = {'Skip1_seed_42': '/kaggle/input/models/lamky690/alpha-1/pytorch/default/1/best_checkpoint.pth',
 'Skip1_seed_142': '/kaggle/input/models/lamky690/142-skip1real/pytorch/default/1/best_checkpoint.pth',
 'Skip0.1_seed42': '/kaggle/input/models/lamky690/skip-newloss/pytorch/default/1/best_checkpoint.pth',
 'Skip0.1_seed142': '/kaggle/input/models/lamky690/142/pytorch/default/1/inv142.pth',
 'Skip0.2_seed42': '/kaggle/input/models/lamky690/0-2alpha42/pytorch/default/1/best_checkpoint.pth', 
 'Skip0.2_seed142': '/kaggle/input/models/lamky690/14-2-skip1/pytorch/default/1/best_checkpoint.pth', #Nham ten model chinh xac la skip 0.2 va seed 142
 'DETR_RoBERTa_42': '/kaggle/input/models/lamky690/bestbert50epoch/pytorch/default/2/best_checkpoint.pth',
 'DETR_RoBERTa_142': '/kaggle/input/models/lamky690/seed142-ftnormal/pytorch/default/1/best_checkpoint.pth'}
RUN_MODELS = list(MODEL_PATHS)  # có thể chọn ít model hơn để tiết kiệm GPU
VAL_MODEL = 'DETR_RoBERTa_42'
SEED_PAIRS = {
    'Baseline': ('DETR_RoBERTa_42', 'DETR_RoBERTa_142'),
    'Skip α = 0.1': ('Skip0.1_seed42', 'Skip0.1_seed142'),
    'Skip α = 0.2': ('Skip0.2_seed42', 'Skip0.2_seed142'),
    'Skip α = 1': ('Skip1_seed_42', 'Skip1_seed_142'),
}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('Bật GPU trong Kaggle Settings trước khi chạy.')


In [2]:
# 2. Dữ liệu và DETR gốc
def keyword_mask(texts, words):
    pattern = r'\b(?:' + '|'.join(re.escape(w) for w in sorted(words)) + r')\b'
    return texts.str.contains(pattern, case=False, regex=True, na=False)

with PERSON_JSON.open(encoding='utf-8') as f:
    person_data = json.load(f)
if not isinstance(person_data, list):
    raise TypeError('person_samples.json phải chứa danh sách mẫu.')

def prepare_split(split):
    rows = []
    for s in person_data:
        if s['split'] != split:
            continue
        w, h = float(s['width']), float(s['height'])
        x, y, bw, bh = map(float, s['bbox'])
        if not np.isfinite([w, h, x, y, bw, bh]).all() or min(w, h, bw, bh) <= 0:
            raise ValueError(f"Kích thước/bbox không hợp lệ: {s['sent_id']}")
        if not isinstance(s['text'], str) or not s['text'].strip():
            raise ValueError(f"Thiếu câu mô tả: {s['sent_id']}")
        filename = Path(s['file_name']).name
        path = next((p for p in [IMAGE_ROOT/filename, IMAGE_ROOT/'train2014'/filename]
                     if p.is_file()), None)
        if path is None:
            raise FileNotFoundError(filename)
        rows.append({
            'sent_id': int(s['sent_id']), 'image_id': int(s['image_id']),
            'text': s['text'], 'image_path': str(path),
            'gt_xyxy': [x/w, y/h, (x+bw)/w, (y+bh)/h],
        })
    df = pd.DataFrame(rows)
    if len(df) != EXPECTED_COUNTS[split] or df['sent_id'].duplicated().any():
        raise ValueError(f'{split}: sai số mẫu hoặc trùng sent_id ({len(df)} mẫu).')
    df['has_color'] = keyword_mask(df['text'], COLOR_WORDS)
    return df

DATA = {split: prepare_split(split) for split in EXPECTED_COUNTS}
for split, df in DATA.items():
    print(f"{split}: {len(df):,} câu | màu: {int(df['has_color'].sum()):,}")
if int(DATA['testA']['has_color'].sum()) != EXPECTED_TESTA_COLOR:
    raise ValueError('Nhóm màu testA khác 1.328 câu; kiểm tra COLOR_WORDS và dữ liệu.')

image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=InterpolationMode.BILINEAR, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

if not DETR_CHECKPOINT.is_file() or not zipfile.is_zipfile(DETR_CHECKPOINT):
    # Chỉ cài gdown khi cần tải; giữ nguyên nguồn tải của notebook gốc.
    import importlib.util
    if importlib.util.find_spec('gdown') is None:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown==5.2.0'])
    import gdown
    DETR_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    temporary = DETR_CHECKPOINT.with_suffix('.download')
    downloaded = gdown.download(id='1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp',
                               output=str(temporary), quiet=False)
    if downloaded is None or not zipfile.is_zipfile(temporary):
        raise RuntimeError('Tải DETR chưa thành công. Kiểm tra Internet trong Kaggle Settings.')
    temporary.replace(DETR_CHECKPOINT)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

detr_sha256 = sha256_file(DETR_CHECKPOINT)
print(f'DETR: {DETR_CHECKPOINT.stat().st_size / 1024**2:.1f} MB')


val: 5,498 câu | màu: 1,503
testA: 5,273 câu | màu: 1,328


Downloading...
From (original): https://drive.google.com/uc?id=1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp
From (redirected): https://drive.google.com/uc?id=1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp&confirm=t&uuid=54c9dd11-a15f-42c7-8424-299325bc5275
To: /kaggle/working/pretrained/detr-r50-unc.download
100%|██████████| 128M/128M [00:01<00:00, 123MB/s]  


DETR: 122.1 MB


In [3]:
# 3. Kiến trúc nguồn — giữ phép tính và tên tham số để nạp strict=True
# Visual modules: TransVG c8624277018fc9786a39181e229f155f933b2b59
# Copyright (c) Facebook, Inc. and its affiliates. All Rights Reserved
from pathlib import Path
import copy
import gc
import math
import os
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoConfig
from transformers.models.roberta.modeling_roberta import RobertaLayer
import argparse
import pickle
import torchvision
from typing import Dict, List, Optional
from torch import Tensor
from torchvision.models._utils import IntermediateLayerGetter

class NestedTensor:

    def __init__(self, tensors, mask):
        self.tensors, self.mask = (tensors, mask)

    def decompose(self):
        return (self.tensors, self.mask)

def nested_tensor_from_tensor_list(images):
    assert isinstance(images, torch.Tensor) and images.ndim == 4
    mask = torch.zeros(images.shape[0], *images.shape[-2:], device=images.device, dtype=torch.bool)
    return NestedTensor(images, mask)

class PositionEmbeddingSine(nn.Module):

    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError('normalize should be True if scale is passed')
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor_list: NestedTensor):
        x = tensor_list.tensors
        mask = tensor_list.mask
        assert mask is not None
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-06
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

class FrozenBatchNorm2d(torch.nn.Module):

    def __init__(self, n):
        super(FrozenBatchNorm2d, self).__init__()
        self.register_buffer('weight', torch.ones(n))
        self.register_buffer('bias', torch.zeros(n))
        self.register_buffer('running_mean', torch.zeros(n))
        self.register_buffer('running_var', torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs):
        num_batches_tracked_key = prefix + 'num_batches_tracked'
        if num_batches_tracked_key in state_dict:
            del state_dict[num_batches_tracked_key]
        super(FrozenBatchNorm2d, self)._load_from_state_dict(state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        eps = 1e-05
        scale = w * (rv + eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias

class BackboneBase(nn.Module):

    def __init__(self, name: str, backbone: nn.Module, num_channels: int, return_interm_layers: bool):
        super().__init__()
        for name, parameter in backbone.named_parameters():
            if 'layer2' not in name and 'layer3' not in name and ('layer4' not in name):
                parameter.requires_grad_(False)
        if return_interm_layers:
            return_layers = {'layer1': '0', 'layer2': '1', 'layer3': '2', 'layer4': '3'}
        else:
            return_layers = {'layer4': '0'}
        self.body = IntermediateLayerGetter(backbone, return_layers=return_layers)
        self.num_channels = num_channels

    def forward(self, tensor_list: NestedTensor):
        xs = self.body(tensor_list.tensors)
        out: Dict[str, NestedTensor] = {}
        for name, x in xs.items():
            m = tensor_list.mask
            assert m is not None
            mask = F.interpolate(m[None].float(), size=x.shape[-2:]).to(torch.bool)[0]
            out[name] = NestedTensor(x, mask)
        return out

class Backbone(BackboneBase):

    def __init__(self, name: str, return_interm_layers: bool, dilation: bool):
        backbone = getattr(torchvision.models, name)(replace_stride_with_dilation=[False, False, dilation], weights=None, norm_layer=FrozenBatchNorm2d)
        assert name in ('resnet50', 'resnet101')
        num_channels = 2048
        super().__init__(name, backbone, num_channels, return_interm_layers)

class Joiner(nn.Sequential):

    def __init__(self, backbone, position_embedding):
        super().__init__(backbone, position_embedding)

    def forward(self, tensor_list: NestedTensor):
        xs = self[0](tensor_list)
        out: List[NestedTensor] = []
        pos = []
        for name, x in xs.items():
            out.append(x)
            pos.append(self[1](x).to(x.tensors.dtype))
        return (out, pos)

class Transformer(nn.Module):

    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1, activation='relu', normalize_before=False, return_intermediate_dec=False):
        super().__init__()
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, activation, normalize_before)
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, encoder_norm)
        if num_decoder_layers > 0:
            decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout, activation, normalize_before)
            decoder_norm = nn.LayerNorm(d_model)
            self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, decoder_norm, return_intermediate=return_intermediate_dec)
        else:
            self.decoder = None
        self._reset_parameters()
        self.d_model = d_model
        self.nhead = nhead

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, pos_embed, query_embed=None):
        bs, c, h, w = src.shape
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        mask = mask.flatten(1)
        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        if self.decoder is not None:
            query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
            tgt = torch.zeros_like(query_embed)
            hs = self.decoder(tgt, memory, memory_key_padding_mask=mask, pos=pos_embed, query_pos=query_embed)
            return (hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w))
        else:
            return (mask, memory)

class TransformerEncoder(nn.Module):

    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src, mask: Optional[Tensor]=None, src_key_padding_mask: Optional[Tensor]=None, pos: Optional[Tensor]=None):
        output = src
        for layer in self.layers:
            output = layer(output, src_mask=mask, src_key_padding_mask=src_key_padding_mask, pos=pos)
        if self.norm is not None:
            output = self.norm(output)
        return output

class TransformerEncoderLayer(nn.Module):

    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1, activation='relu', normalize_before=False):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(self, src, src_mask: Optional[Tensor]=None, src_key_padding_mask: Optional[Tensor]=None, pos: Optional[Tensor]=None):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(self, src, src_mask: Optional[Tensor]=None, src_key_padding_mask: Optional[Tensor]=None, pos: Optional[Tensor]=None):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(q, k, value=src2, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(self, src, src_mask: Optional[Tensor]=None, src_key_padding_mask: Optional[Tensor]=None, pos: Optional[Tensor]=None):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)

def _get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

def _get_activation_fn(activation):
    if activation == 'relu':
        return F.relu
    if activation == 'gelu':
        return F.gelu
    if activation == 'glu':
        return F.glu
    raise RuntimeError(f'activation should be relu/gelu, not {activation}.')

class DETR(nn.Module):

    def __init__(self, backbone, transformer, num_queries, train_backbone, train_transformer, aux_loss=False):
        super().__init__()
        self.num_queries = num_queries
        self.transformer = transformer
        self.backbone = backbone
        if self.transformer is not None:
            hidden_dim = transformer.d_model
            self.input_proj = nn.Conv2d(backbone.num_channels, hidden_dim, kernel_size=1)
        else:
            hidden_dim = backbone.num_channels
        if not train_backbone:
            for p in self.backbone.parameters():
                p.requires_grad_(False)
        if self.transformer is not None and (not train_transformer):
            for m in [self.transformer, self.input_proj]:
                for p in m.parameters():
                    p.requires_grad_(False)
        self.num_channels = hidden_dim

    def forward(self, samples: NestedTensor):
        if isinstance(samples, (list, torch.Tensor)):
            samples = nested_tensor_from_tensor_list(samples)
        features, pos = self.backbone(samples)
        src, mask = features[-1].decompose()
        assert mask is not None
        if self.transformer is not None:
            out = self.transformer(self.input_proj(src), mask, pos[-1], query_embed=None)
        else:
            out = [mask.flatten(1), src.flatten(2).permute(2, 0, 1)]
        return out

class FrozenDETRFeatures(nn.Module):

    def __init__(self, checkpoint):
        super().__init__()
        backbone = Backbone('resnet50', return_interm_layers=False, dilation=False)
        position = PositionEmbeddingSine(128, normalize=True)
        joiner = Joiner(backbone, position)
        joiner.num_channels = backbone.num_channels
        transformer = Transformer(d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=0, dim_feedforward=2048, dropout=0.1, activation='relu', normalize_before=False)
        self.visual = DETR(joiner, transformer, num_queries=100, train_backbone=False, train_transformer=False)
        with torch.serialization.safe_globals([argparse.Namespace]):
            payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
        state = payload.get('model', payload)
        assert isinstance(state, dict) and state, 'Checkpoint thiếu model state_dict.'
        state = {k.removeprefix('module.'): v for k, v in state.items()}
        required = self.visual.state_dict()
        missing = sorted(set(required) - set(state))
        assert not missing, f'Checkpoint thiếu visual weights: {missing[:20]}'
        mismatched = [k for k in required if tuple(required[k].shape) != tuple(state[k].shape)]
        assert not mismatched, f'Backbone/cấu hình checkpoint không khớp: {mismatched[:20]}'
        extra = sorted(set(state) - set(required))
        allowed = ('transformer.decoder.', 'class_embed.', 'bbox_embed.', 'query_embed.')
        unexpected = [k for k in extra if not k.startswith(allowed)]
        assert not unexpected, f'Checkpoint có visual keys không nhận diện: {unexpected[:20]}'
        self.visual.load_state_dict({k: state[k] for k in required}, strict=True)
        print('Loaded ALL backbone + input_proj + 6-layer encoder weights;', len(extra), 'unused detection decoder/head keys.')
        del state, payload
        self.requires_grad_(False)
        self.eval()

    def train(self, mode=True):
        super().train(False)
        return self

    @torch.no_grad()
    @torch.autocast(device_type='cuda', dtype=torch.float16)
    def forward(self, images):
        padding, memory = self.visual(nested_tensor_from_tensor_list(images))
        h, w = (math.ceil(images.shape[-2] / 32), math.ceil(images.shape[-1] / 32))
        assert tuple(memory.shape) == (h * w, len(images), 256)
        assert not padding.any(), 'Ảnh đã resize cùng size nên mask phải toàn False.'
        return memory.permute(1, 2, 0).reshape(len(images), 256, h, w).contiguous()

class PositionEncoding2D(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        assert d_model % 4 == 0
        self.d_model = d_model

    def forward(self, feature_map):
        _, _, H, W = feature_map.shape
        device = feature_map.device
        n = self.d_model // 4
        frequencies = 10000 ** (torch.arange(n, device=device, dtype=torch.float32) / n)
        y = torch.linspace(0, 2 * math.pi, H, device=device)
        x = torch.linspace(0, 2 * math.pi, W, device=device)
        y = y[:, None] / frequencies[None, :]
        x = x[:, None] / frequencies[None, :]
        pos_y = torch.cat([y.sin(), y.cos()], dim=-1)
        pos_x = torch.cat([x.sin(), x.cos()], dim=-1)
        pos_y = pos_y[:, None, :].expand(H, W, -1)
        pos_x = pos_x[None, :, :].expand(H, W, -1)
        position = torch.cat([pos_y, pos_x], dim=-1)
        return position.reshape(1, H * W, self.d_model).to(dtype=feature_map.dtype)

class DETRCachedREC(nn.Module):

    def __init__(self, d_model=256):
        super().__init__()
        self.image_projection = nn.Conv2d(256, d_model, 1)
        self.image_position = PositionEncoding2D(d_model)
        self.text_projection = nn.Linear(768, d_model)
        self.modality_embedding = nn.Embedding(2, d_model)
        self.transformer = nn.Transformer(d_model=d_model, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=1024, dropout=0.1, activation='relu', batch_first=True)
        self.object_query = nn.Embedding(1, d_model)
        self.box_head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 4), nn.Sigmoid())

    @torch.autocast(device_type='cuda', dtype=torch.float16)
    def forward(self, image_features, text_features, attention_mask):
        assert image_features.shape[1] == 256, 'Cần spatial memory DETR encoder.'
        x = image_features
        x = self.image_projection(x)
        image_tokens = x.flatten(2).transpose(1, 2)
        image_tokens = image_tokens + self.image_position(x) + self.modality_embedding.weight[0]
        text_tokens = self.text_projection(text_features) + self.modality_embedding.weight[1]
        tokens = torch.cat([image_tokens, text_tokens], dim=1)
        image_padding = torch.zeros(image_features.shape[0], image_tokens.shape[1], dtype=torch.bool, device=image_features.device)
        padding = torch.cat([image_padding, ~attention_mask.bool()], dim=1)
        memory = self.transformer.encoder(tokens, src_key_padding_mask=padding)
        query = self.object_query.weight.unsqueeze(0).expand(image_features.shape[0], -1, -1)
        decoded = self.transformer.decoder(query, memory, memory_key_padding_mask=padding)
        return self.box_head(decoded[:, 0, :])

class FineTuneREC(DETRCachedREC):

    def __init__(self, visual, text_tail):
        super().__init__()
        self.layer4 = visual.backbone[0].body.layer4
        self.pos = visual.backbone[1]
        self.skip_alpha = SKIP_ALPHA
        self.proj = visual.input_proj
        self.visual_encoder = visual.transformer.encoder
        self.text_tail = text_tail

    @torch.autocast('cuda', dtype=torch.float16)
    def forward(self, image, text, attention_mask, return_features=False):
        x = self.proj(self.layer4(image))
        nested = nested_tensor_from_tensor_list(x)
        pos = self.pos(nested).to(x.dtype).flatten(2).permute(2, 0, 1)
        tokens = x.flatten(2).permute(2, 0, 1)
        memory = self.visual_encoder(tokens, src_key_padding_mask=nested.mask.flatten(1), pos=pos)
        x_post = memory.permute(1, 2, 0).reshape_as(x)
        image = x_post + self.skip_alpha * x if self.skip_alpha else x_post
        mask = (~attention_mask)[:, None, None, :].to(text.dtype) * torch.finfo(text.dtype).min
        for layer in self.text_tail:
            output = layer(text, attention_mask=mask)
            text = output[0] if isinstance(output, (tuple, list)) else output
            if text.ndim != 3:
                raise RuntimeError(f'Text phải có shape [batch, tokens, hidden], nhận {tuple(text.shape)}')
        boxes = super().forward(image, text, attention_mask)
        return (boxes, image, text) if return_features else boxes


In [4]:
# 4. Loader — mỗi lần chỉ đưa một model lên GPU
_visual_template = None
_text_prefix_cache = {}

def get_visual_template():
    global _visual_template
    if _visual_template is None:
        source = FrozenDETRFeatures(DETR_CHECKPOINT)
        _visual_template = source.visual.cpu().eval()
    return _visual_template

def get_text_prefix_and_tokenizer(text_model, revision):
    key = (text_model, revision)
    if key not in _text_prefix_cache:
        tokenizer_local = AutoTokenizer.from_pretrained(text_model, revision=revision)
        prefix = AutoModel.from_pretrained(text_model, revision=revision, add_pooling_layer=False, attn_implementation='eager')
        if len(prefix.encoder.layer) != 12:
            raise ValueError('Đoạn code này yêu cầu RoBERTa 12 layer.')
        prefix.encoder.layer = nn.ModuleList(list(prefix.encoder.layer)[:10])
        prefix.config.num_hidden_layers = 10
        prefix.requires_grad_(False).cpu().eval()
        _text_prefix_cache[key] = (prefix, tokenizer_local)
    return _text_prefix_cache[key]

class EvalFullREC(nn.Module):

    def __init__(self, stem, prefix, rec):
        super().__init__()
        self.stem = stem
        self.prefix = prefix
        self.rec = rec

    def forward(self, images, input_ids, attention_mask):
        image_features = self.stem(images)
        text_features = self.prefix(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).last_hidden_state
        return self.rec(image_features, text_features, attention_mask.bool())

def load_eval_model(name, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    payload = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    cfg = dict(payload['config'])
    state = {k.removeprefix('module.'): v for k, v in payload['model_state_dict'].items()}
    supported_stages = {'rec_finetune_layer4_text2', 'rec_finetune_layer4_text2_skip', 'rec_finetune_layer4_text2_skip_contrastive'}
    stage = cfg.get('stage')
    if stage not in supported_stages:
        raise ValueError(f'{name}: stage chưa hỗ trợ: {stage!r}')
    if cfg.get('detr_model') != 'TransVG/detr-r50-unc':
        raise ValueError(f'{name}: sai loại DETR.')
    if 'skip' in stage and 'skip_alpha' not in cfg:
        raise ValueError(f'{name}: checkpoint skip thiếu skip_alpha.')
    alpha = float(cfg.get('skip_alpha', 0.0))
    if not math.isfinite(alpha):
        raise ValueError(f'{name}: skip_alpha không hợp lệ.')
    if name.startswith('DETR_RoBERTa') and alpha != 0.0:
        raise ValueError(f'{name}: cần checkpoint không skip nhưng alpha={alpha}.')
    if name.startswith('Skip') and alpha == 0.0:
        raise ValueError(f'{name}: checkpoint đang có alpha=0.')
    expected_alpha = re.match('Skip([0-9.]+)', name)
    if expected_alpha and (not math.isclose(alpha, float(expected_alpha.group(1)), abs_tol=1e-08)):
        raise ValueError(f'{name}: alpha trong checkpoint = {alpha}, khác tên cấu hình.')
    text_model = cfg.get('text_model', 'FacebookAI/roberta-base')
    revision = cfg.get('text_revision')
    if not revision:
        raise ValueError(f'{name}: checkpoint thiếu text_revision.')
    expected_hash = cfg.get('detr_sha256')
    if expected_hash:
        import hashlib
        actual_hash = globals().get('detr_sha256')
        if actual_hash is None:
            digest = hashlib.sha256()
            with Path(DETR_CHECKPOINT).open('rb') as f:
                for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
                    digest.update(block)
            actual_hash = digest.hexdigest()
        if actual_hash != expected_hash:
            raise ValueError(f'{name}: DETR gốc không trùng SHA256.')
    visual = copy.deepcopy(get_visual_template())
    stem = nn.Sequential(*list(visual.backbone[0].body.children())[:-1])
    stem.requires_grad_(False)
    text_cfg = AutoConfig.from_pretrained(text_model, revision=revision)
    text_cfg._attn_implementation = 'eager'
    tail = nn.ModuleList([RobertaLayer(text_cfg) for _ in range(2)])
    rec = FineTuneREC(visual, tail)
    rec.skip_alpha = alpha
    rec.load_state_dict(state, strict=True)
    prefix, tokenizer_local = get_text_prefix_and_tokenizer(text_model, revision)
    full_model = EvalFullREC(stem, prefix, rec)
    full_model.requires_grad_(False).to(device).eval()
    if cfg.get('channels_last', False):
        full_model.stem.to(memory_format=torch.channels_last)
        full_model.rec.layer4.to(memory_format=torch.channels_last)
    info = {'epoch': payload.get('epoch'), 'best_val': payload.get('best_acc50'), 'skip_alpha': alpha, 'contrastive_alpha': float(cfg.get('contrastive_alpha', 0.0))}
    del payload, state, visual
    gc.collect()
    return (full_model, tokenizer_local, cfg, info)


Khoi phuc model

In [5]:
# 5. Đánh giá dùng chung và cache IoU từng câu
MODEL_INFO = {}
PREDICTIONS = {}

def paired_iou(pred, gt):
    center, half = pred[:, :2], pred[:, 2:] / 2
    pred = torch.cat([center-half, center+half], dim=1)
    intersection = (torch.minimum(pred[:, 2:], gt[:, 2:]) -
                    torch.maximum(pred[:, :2], gt[:, :2])).clamp_min(0).prod(1)
    areas = [(box[:, 2:] - box[:, :2]).clamp_min(0).prod(1) for box in (pred, gt)]
    return intersection / (areas[0] + areas[1] - intersection).clamp_min(1e-8)

def validate_predictions(df, metadata):
    if df['sent_id'].duplicated().any() or set(df['sent_id']) != set(metadata['sent_id']):
        raise ValueError('Cache dự đoán có sent_id trùng hoặc không khớp dữ liệu.')
    if not df['iou'].between(0, 1).all():
        raise ValueError('IoU bị thiếu hoặc ngoài [0, 1].')
    return metadata.merge(df[['sent_id', 'iou']], on='sent_id', validate='one_to_one')

def evaluate(name, split='testA'):
    metadata = DATA[split]
    checkpoint = Path(MODEL_PATHS[name])
    stat = checkpoint.stat()
    signature = {
        'version': EVAL_VERSION, 'checkpoint': str(checkpoint),
        'size': stat.st_size, 'mtime': stat.st_mtime_ns,
        'detr': detr_sha256, 'image_size': IMAGE_SIZE,
        'text_length': MAX_TEXT_LENGTH, 'batch_size': BATCH_SIZE, 'colors': sorted(COLOR_WORDS),
        'torch': torch.__version__, 'transformers': __import__('transformers').__version__,
        'torchvision': __import__('torchvision').__version__,
        'samples': metadata[['sent_id', 'image_id', 'text', 'image_path', 'gt_xyxy']].to_json(orient='records'),
    }
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:20]
    cache = OUT_DIR / f'{name}_{split}_{digest}.csv'
    info_path = cache.with_suffix('.json')
    key = (name, split, digest)
    if not FORCE_EVAL and key in PREDICTIONS:
        return PREDICTIONS[key].copy()
    if not FORCE_EVAL and cache.is_file() and info_path.is_file():
        result = validate_predictions(pd.read_csv(cache), metadata)
        MODEL_INFO[name] = json.loads(info_path.read_text())
        PREDICTIONS[key] = result
        print(f'Đọc cache: {name} / {split}')
        return result.copy()

    model = None
    try:
        model, tokenizer, cfg, info = load_eval_model(name, checkpoint)
        print(f"{name} / {split} | epoch={info['epoch']} | α={info['skip_alpha']}")
        rows = metadata.to_dict('records')
        values = []
        with torch.inference_mode():
            for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=f'{split} — {name}'):
                batch = rows[start:start+BATCH_SIZE]
                images = []
                for sample in batch:
                    with Image.open(sample['image_path']) as image:
                        images.append(image_transform(image.convert('RGB')))
                images = torch.stack(images).to(device)
                gt = torch.tensor([s['gt_xyxy'] for s in batch], dtype=torch.float32, device=device)
                tokens = tokenizer([s['text'] for s in batch], padding=True, truncation=True,
                                   max_length=MAX_TEXT_LENGTH, return_tensors='pt').to(device)
                # Kiến trúc nguồn đã dùng autocast FP16; giữ cùng chế độ đánh giá.
                with torch.autocast('cuda', dtype=torch.float16):
                    pred = model(images, tokens['input_ids'], tokens['attention_mask'])
                if tuple(pred.shape) != (len(batch), 4) or not torch.isfinite(pred).all():
                    raise ValueError('Hộp dự đoán phải là tensor hữu hạn [B, 4].')
                values.extend(paired_iou(pred.float(), gt).cpu().tolist())
        predicted = pd.DataFrame({'sent_id': metadata['sent_id'], 'iou': values})
        result = validate_predictions(predicted, metadata)
        temporary = cache.with_suffix('.tmp')
        predicted.to_csv(temporary, index=False)
        temporary.replace(cache)
        info_path.write_text(json.dumps(info, default=str, indent=2))
        MODEL_INFO[name] = info
        PREDICTIONS[key] = result
        return result.copy()
    finally:
        if model is not None:
            model.cpu()  # prefix dùng chung được chuyển về CPU trước khi giải phóng
            del model
        gc.collect()
        torch.cuda.empty_cache()

def metric_row(name, subset, df):
    return {
        'Model': name, 'Tập': subset, 'N': len(df), **MODEL_INFO[name],
        'Acc@0.5 (%)': 100 * df['iou'].ge(0.5).mean(),
        'Acc@0.75 (%)': 100 * df['iou'].ge(0.75).mean(),
        'mIoU': df['iou'].mean(),
    }


In [6]:
# 6. testA: từng model và trung bình ± SD của hai seed
rows = []
for name in RUN_MODELS:
    df = evaluate(name, 'testA')
    rows.append(metric_row(name, 'testA người', df))
    rows.append(metric_row(name, 'testA người + màu', df.loc[df['has_color']]))
results_df = pd.DataFrame(rows)
results_df.to_csv(OUT_DIR / 'results_testA.csv', index=False)

metrics = ['Acc@0.5 (%)', 'Acc@0.75 (%)', 'mIoU']
for subset, table in results_df.groupby('Tập', sort=False):
    print('\n' + subset)
    display(table.set_index('Model')[['N', 'epoch', 'skip_alpha', 'contrastive_alpha'] + metrics]
            .style.format({metrics[0]: '{:.2f}', metrics[1]: '{:.2f}', 'mIoU': '{:.4f}'}))

pair_rows = []
for subset, table in results_df.groupby('Tập', sort=False):
    table = table.set_index('Model')
    for pair, members in SEED_PAIRS.items():
        if not set(members).issubset(table.index):
            continue
        values = table.loc[list(members)]
        if values['N'].nunique() != 1:
            raise ValueError(f'{pair}: số mẫu giữa hai seed không khớp.')
        row = {'Cặp': pair, 'Tập': subset, 'N': int(values['N'].iloc[0])}
        for metric in metrics:
            row[metric + ' mean'] = values[metric].mean()
            row[metric + ' std'] = values[metric].std(ddof=1)
        pair_rows.append(row)
pair_results_df = pd.DataFrame(pair_rows)
pair_results_df.to_csv(OUT_DIR / 'results_two_seeds.csv', index=False)
if not pair_results_df.empty:
    for subset, table in pair_results_df.groupby('Tập', sort=False):
        shown = table[['Cặp', 'N']].copy()
        for metric in metrics:
            digits = 4 if metric == 'mIoU' else 2
            shown[metric] = [f'{mean:.{digits}f} ± {std:.{digits}f}'
                             for mean, std in zip(table[metric+' mean'], table[metric+' std'])]
        print(f'\n{subset} — trung bình ± SD mẫu (n = 2)')
        display(shown.set_index('Cặp'))


Loaded ALL backbone + input_proj + 6-layer encoder weights; 0 unused detection decoder/head keys.


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (24 outputs left out)


testA người — trung bình ± SD mẫu (n = 2)


,N,Acc@0.5 (%),Acc@0.75 (%),mIoU
Cặp,,,,
Baseline,5273,83.35 ± 0.67,69.49 ± 0.91,0.7298 ± 0.0037
Skip α = 0.1,5273,83.92 ± 0.19,70.75 ± 0.55,0.7354 ± 0.0023
Skip α = 0.2,5273,84.12 ± 0.31,70.89 ± 0.19,0.7389 ± 0.0008
Skip α = 1,5273,84.77 ± 0.03,71.08 ± 0.08,0.7395 ± 0.0015



testA người + màu — trung bình ± SD mẫu (n = 2)


,N,Acc@0.5 (%),Acc@0.75 (%),mIoU
Cặp,,,,
Baseline,1328,76.58 ± 1.28,61.48 ± 1.22,0.6717 ± 0.0080
Skip α = 0.1,1328,79.82 ± 0.43,64.95 ± 1.33,0.6948 ± 0.0069
Skip α = 0.2,1328,80.69 ± 1.76,66.53 ± 0.16,0.7078 ± 0.0082
Skip α = 1,1328,82.57 ± 0.27,67.70 ± 1.70,0.7150 ± 0.0011


In [7]:
# 7. Phân tích thuộc tính trên val — tái sử dụng cache nếu đã đánh giá
ATTRIBUTE_WORDS = {
    "Giới tính": {
        "man", "men", "woman", "women", "male", "female",
        "boy", "boys", "girl", "girls", "guy", "guys",
        "lady", "ladies", "gentleman", "gentlemen",
    },
    "Màu sắc": set(COLOR_WORDS),
    "Phụ kiện": {
        "hat", "cap", "helmet", "glasses", "sunglasses",
        "bag", "backpack", "purse", "scarf", "tie",
        "watch", "necklace", "earrings",
    },
    "Họa tiết": {
        "striped", "stripe", "stripes", "stripy",
        "plaid", "checked", "checkered", "chequered",
        "polka dot", "polka dots", "dotted",
        "floral", "spotted", "camouflage",
    },
    "Kiểu tóc": {
        "hair", "haired", "bald", "blond", "blonde",
        "brunette", "ponytail", "braid", "braids",
    },
    "Tuổi": {
        "young", "younger", "old", "older", "elderly",
        "kid", "kids", "child", "children",
        "baby", "babies", "teen", "teenager",
        "adult", "adults",
    },
}
df_val_baseline_42 = evaluate(VAL_MODEL, 'val')
def attribute_row(name, df):
    n = len(df)
    return {'Thuộc tính': name, 'Số câu': n,
            'Số câu sai': int(df['iou'].lt(0.5).sum()),
            'Acc@0,5 (%)': 100*df['iou'].ge(0.5).mean() if n else np.nan}

rows = [attribute_row('Toàn bộ val', df_val_baseline_42)]
for name, words in ATTRIBUTE_WORDS.items():
    mask = keyword_mask(df_val_baseline_42['text'], words)
    rows.append(attribute_row(name, df_val_baseline_42.loc[mask]))
attribute_summary = pd.DataFrame(rows)
attribute_summary.to_csv(OUT_DIR / 'val_attributes.csv', index=False)
print(f'{VAL_MODEL} — các nhóm thuộc tính có thể giao nhau.')
display(attribute_summary.style.format({'Acc@0,5 (%)': '{:.2f}'}, na_rep='—'))


DETR_RoBERTa_42 / val | epoch=42 | α=0.0


val — DETR_RoBERTa_42:   0%|          | 0/344 [00:00<?, ?it/s]

DETR_RoBERTa_42 — các nhóm thuộc tính có thể giao nhau.


,Thuộc tính,Số câu,Số câu sai,"Acc@0,5 (%)"
0,Toàn bộ val,5498,904,83.56
1,Giới tính,3059,477,84.41
2,Màu sắc,1503,323,78.51
3,Phụ kiện,211,55,73.93
4,Họa tiết,103,45,56.31
5,Kiểu tóc,90,21,76.67
6,Tuổi,293,53,81.91


In [9]:
# 8. McNemar — 2 baseline × 2 skip α = 1, không chạy lại model đã có cache
BASELINES = {42: 'DETR_RoBERTa_42', 142: 'DETR_RoBERTa_142'}
PROPOSED = {42: 'Skip1_seed_42', 142: 'Skip1_seed_142'}

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty(len(p))
    adjusted[order] = np.minimum(1, np.maximum.accumulate(p[order] * np.arange(len(p), 0, -1)))
    return adjusted

def compare_checkpoints(predictions, reference, same_seed_only=False):
    ids = reference['sent_id'].to_numpy()
    columns = {}
    for name, df in predictions.items():
        if df['sent_id'].duplicated().any() or set(df['sent_id']) != set(ids):
            raise ValueError(f'{name}: sent_id trùng hoặc không khớp.')
        if not df['iou'].between(0, 1).all():
            raise ValueError(f'{name}: IoU không hợp lệ.')
        columns[name] = df.set_index('sent_id')['iou'].reindex(ids).ge(0.5)
    correct = pd.DataFrame(columns)
    color_ids = set(reference.loc[reference['has_color'], 'sent_id'])
    groups = {'testA người': correct,
              'testA người + màu': correct.loc[correct.index.isin(color_ids)]}
    pairs = [(42, 42), (142, 142)] if same_seed_only else list(product([42, 142], repeat=2))
    rows = []
    for subset, df in groups.items():
        if df.empty:
            raise ValueError(f'{subset}: tập rỗng.')
        for sa, sb in pairs:
            a, b = df[BASELINES[sa]], df[PROPOSED[sb]]
            lose, gain = int((a & ~b).sum()), int((~a & b).sum())
            p = binomtest(gain, gain+lose, p=0.5, alternative='two-sided').pvalue if gain+lose else 1.0
            rows.append({'Tập': subset, 'Baseline seed': sa, 'Skip seed': sb, 'N': len(df),
                         'Cả hai đúng': int((a & b).sum()), 'Baseline đúng riêng': lose,
                         'Skip đúng riêng': gain, 'Cả hai sai': int((~a & ~b).sum()),
                         'Acc baseline (%)': 100*a.mean(), 'Acc skip (%)': 100*b.mean(),
                         'Chênh lệch (điểm %)': 100*(b.mean()-a.mean()), 'p_exact': p})
    result = pd.DataFrame(rows)
    result['p_holm'] = holm_adjust(result['p_exact'])
    result['Skip tốt hơn (p_Holm < 0.05)'] = result['p_holm'].lt(0.05) & result['Chênh lệch (điểm %)'].gt(0)
    return result

test_predictions = {name: evaluate(name, 'testA')
                    for name in list(BASELINES.values()) + list(PROPOSED.values())}
results_mcnemar = compare_checkpoints(test_predictions, DATA['testA'], SAME_SEED_ONLY)
results_mcnemar.to_csv(OUT_DIR / 'mcnemar.csv', index=False)
print(f'McNemar exact hai phía; Holm chung cho {len(results_mcnemar)} phép thử.')
display(results_mcnemar.style.format({
    'Acc baseline (%)': '{:.2f}', 'Acc skip (%)': '{:.2f}',
    'Chênh lệch (điểm %)': '{:+.2f}', 'p_exact': '{:.6g}', 'p_holm': '{:.6g}',
}))


McNemar exact hai phía; Holm chung cho 8 phép thử.


,Tập,Baseline seed,Skip seed,N,Cả hai đúng,Baseline đúng riêng,Skip đúng riêng,Cả hai sai,Acc baseline (%),Acc skip (%),Chênh lệch (điểm %),p_exact,p_holm,Skip tốt hơn (p_Holm < 0.05)
0,testA người,42,42,5273,4169,201,300,603,82.88,84.75,+1.88,1.12449e-05,4.40394e-05,True
1,testA người,42,142,5273,4169,201,302,601,82.88,84.79,+1.92,7.71551e-06,3.85775e-05,True
2,testA người,142,42,5273,4174,246,295,558,83.82,84.75,+0.93,0.0389495,0.0490415,True
3,testA người,142,142,5273,4198,222,273,580,83.82,84.79,+0.97,0.0245207,0.0490415,True
4,testA người + màu,42,42,1328,950,55,144,179,75.68,82.38,+6.70,2.22451e-10,1.55716e-09,True
5,testA người + màu,42,142,1328,954,51,145,178,75.68,82.76,+7.08,1.23651e-11,9.89212e-11,True
6,testA người + màu,142,42,1328,954,75,140,159,77.48,82.38,+4.89,1.10099e-05,4.40394e-05,True
7,testA người + màu,142,142,1328,967,62,132,167,77.48,82.76,+5.27,5.54414e-07,3.32648e-06,True
